In [1]:
"""
================================================================================
AHNN — Adaptive Hybrid Neural Network
COMPLETE DATA STRUCTURE CODE (Thesis-aligned, Hinglish comments)
================================================================================
 
Yeh file poore AHNN project ka data structure define karti hai:
  - Constants (N, T, F, folds, groups, etc.)
  - Raw balance-sheet schema
  - 11 financial ratios (feature engineering)
  - Altman Z-score label construction
  - Tensor shapes: X in R^{N x T x F}, y in {0,1}^N
  - Train / Test / Gap validation splits
  - k-Fold cross-validation indices
  - Fairness group (small vs large) assignment
  - Bagging bootstrap indices (7 members)
  - Standardization (StandardScaler) fit on train only
  - Full dataset container class `AHNNDataset`
 
Run:  python ahnn_data_structure.py
================================================================================
"""
 
from __future__ import annotations
 
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
 
 
# ==============================================================================
# 1. GLOBAL CONSTANTS — Thesis ke hisaab se fixed
# ==============================================================================
 
# --- Dataset dimensions ---
N_COMPANIES: int = 62        # N — total companies (thesis: N=62)
T_YEARS: int = 8             # T — time steps per company (thesis: T=8 years)
F_FEATURES: int = 11         # F — financial ratios per year
 
# --- Time window ---
YEAR_START: int = 2016       # first reporting year
YEAR_END: int = 2023         # last reporting year (inclusive) -> 8 years total
ASSERT_YEARS = (YEAR_END - YEAR_START + 1 == T_YEARS)
 
# --- Altman Z-score threshold ---
ALTMAN_DISTRESS_CUTOFF: float = 1.81   # Z < 1.81 => default (y=1)
 
# --- Validation protocol ---
K_FOLDS: int = 5             # k-fold CV
GAP_FRACTION: float = 0.10   # 10% gap for time-aware validation
RANDOM_SEED: int = 42        # reproducibility
 
# --- Model hyperparameters (data-structure ke liye relevant) ---
ENSEMBLE_MEMBERS: int = 7    # bagging ensemble size
FAIRNESS_LAMBDA: float = 0.5
SPECTRAL_BOUND_RHO: float = 3.0
WEIGHT_DECAY: float = 1e-3
DROPOUT_P: float = 0.1
LEARNING_RATE: float = 0.05
EPOCHS: int = 400
 
# --- Fairness groups ---
GROUP_LABELS = ("small", "large")   # median-split on total assets
 
 
# ==============================================================================
# 2. RAW BALANCE-SHEET SCHEMA
# ==============================================================================
# Har company ke har year ke liye yeh columns raw data me chahiye.
# Unit: local currency (consistent across all entries).
 
RAW_COLUMNS: List[str] = [
    "company_id",        # unique company identifier (1..N)
    "company_name",      # readable name
    "year",              # reporting year (YEAR_START..YEAR_END)
    "CA",                # Current Assets
    "CL",                # Current Liabilities
    "TA",                # Total Assets
    "TL",                # Total Liabilities
    "Inventory",         # Inventory
    "Equity",            # Shareholders' Equity
    "Revenue",           # Net Sales / Revenue
    "GrossProfit",       # Gross Profit
    "EBIT",              # Earnings Before Interest & Taxes
    "NetIncome",         # Net Income (after tax)
    "InterestExpense",   # Interest Expense
    "RetainedEarnings",  # Retained Earnings
]
 
 
# ==============================================================================
# 3. FEATURE NAMES (11 ratios — thesis Table, Section 2)
# ==============================================================================
 
FEATURE_NAMES: List[str] = [
    "WC_TA",     # (CA - CL) / TA       — Working capital efficiency
    "RE_TA",     # RetainedEarnings / TA — thesis me NI/TA likha hai, hum RE/TA use karenge (Altman original)
    "EBIT_TA",   # EBIT / TA             — Operating efficiency
    "EQ_TL",     # Equity / TL           — Capital structure
    "S_TA",      # Revenue / TA          — Asset turnover
    "CR",        # CA / CL               — Current ratio
    "QR",        # (CA - Inv) / CL       — Quick ratio
    "NPM",       # NetIncome / Revenue   — Net profit margin
    "GPM",       # GrossProfit / Revenue — Gross margin
    "Lev",       # TL / TA               — Leverage
    "ICR",       # EBIT / InterestExpense — Interest coverage
]
assert len(FEATURE_NAMES) == F_FEATURES, "Feature count mismatch with F=11"
 
 
# ==============================================================================
# 4. FEATURE ENGINEERING — Raw -> 11 ratios
# ==============================================================================
 
def compute_features(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Raw balance-sheet dataframe se 11 ratios compute karta hai.
    Division-by-zero safe: small epsilon add karte hain denominators me.
    """
    eps = 1e-9
    df = raw.copy()
 
    df["WC_TA"]   = (df["CA"] - df["CL"]) / (df["TA"] + eps)
    df["RE_TA"]   = df["RetainedEarnings"] / (df["TA"] + eps)
    df["EBIT_TA"] = df["EBIT"] / (df["TA"] + eps)
    df["EQ_TL"]   = df["Equity"] / (df["TL"] + eps)
    df["S_TA"]    = df["Revenue"] / (df["TA"] + eps)
    df["CR"]      = df["CA"] / (df["CL"] + eps)
    df["QR"]      = (df["CA"] - df["Inventory"]) / (df["CL"] + eps)
    df["NPM"]     = df["NetIncome"] / (df["Revenue"] + eps)
    df["GPM"]     = df["GrossProfit"] / (df["Revenue"] + eps)
    df["Lev"]     = df["TL"] / (df["TA"] + eps)
    df["ICR"]     = df["EBIT"] / (df["InterestExpense"] + eps)
 
    keep = ["company_id", "company_name", "year"] + FEATURE_NAMES
    return df[keep].copy()
 
 
# ==============================================================================
# 5. ALTMAN Z-SCORE LABEL
# ==============================================================================
 
def compute_altman_z(features_df: pd.DataFrame) -> pd.Series:
    """
    Z = 1.2*WC/TA + 1.4*RE/TA + 3.3*EBIT/TA + 0.6*EQ/TL + 1.0*S/TA
    Thesis Section 2.1.
    """
    z = (
        1.2 * features_df["WC_TA"]
        + 1.4 * features_df["RE_TA"]
        + 3.3 * features_df["EBIT_TA"]
        + 0.6 * features_df["EQ_TL"]
        + 1.0 * features_df["S_TA"]
    )
    return z
 
 
def construct_labels(features_df: pd.DataFrame,
                     label_year: Optional[int] = None) -> pd.DataFrame:
    """
    Company-level label: default (1) agar last-year Z < 1.81, else 0.
    Default: label_year = YEAR_END (prediction target = last observed year).
 
    Returns: DataFrame[company_id, y, z_label_year]
    """
    if label_year is None:
        label_year = YEAR_END
 
    last = features_df[features_df["year"] == label_year].copy()
    last["Z"] = compute_altman_z(last)
    last["y"] = (last["Z"] < ALTMAN_DISTRESS_CUTOFF).astype(int)
    return last[["company_id", "y", "Z"]].rename(columns={"Z": "Z_label_year"})
 
 
# ==============================================================================
# 6. TENSOR ASSEMBLY — DataFrame -> X [N,T,F], y [N]
# ==============================================================================
 
def build_tensors(
    features_df: pd.DataFrame,
    labels_df: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    """
    Panel DataFrame ko 3D tensor me convert karta hai.
      X: shape (N, T, F)
      y: shape (N,)
      company_order: list of company_ids in row order
    """
    company_ids = sorted(features_df["company_id"].unique().tolist())
    assert len(company_ids) == N_COMPANIES, \
        f"Expected {N_COMPANIES} companies, got {len(company_ids)}"
 
    years = sorted(features_df["year"].unique().tolist())
    assert len(years) == T_YEARS, f"Expected {T_YEARS} years, got {len(years)}"
 
    X = np.zeros((N_COMPANIES, T_YEARS, F_FEATURES), dtype=np.float32)
 
    for i, cid in enumerate(company_ids):
        sub = features_df[features_df["company_id"] == cid].sort_values("year")
        assert len(sub) == T_YEARS, f"Company {cid} has {len(sub)} years, need {T_YEARS}"
        X[i] = sub[FEATURE_NAMES].to_numpy(dtype=np.float32)
 
    # Align labels with company_order
    label_map = dict(zip(labels_df["company_id"], labels_df["y"]))
    y = np.array([label_map[c] for c in company_ids], dtype=np.int64)
 
    return X, y, company_ids
 
 
# ==============================================================================
# 7. FAIRNESS GROUP ASSIGNMENT (small vs large — median split on TA)
# ==============================================================================
 
def assign_fairness_groups(
    raw: pd.DataFrame,
    company_order: List[int],
    split_year: Optional[int] = None,
) -> np.ndarray:
    """
    Median TA ke upar -> 'large' (1), niche -> 'small' (0).
    split_year default: YEAR_END.
    Returns: np.ndarray shape (N,), values in {0, 1}.
    """
    if split_year is None:
        split_year = YEAR_END
 
    snap = raw[raw["year"] == split_year][["company_id", "TA"]].copy()
    median_ta = snap["TA"].median()
    snap["group"] = (snap["TA"] >= median_ta).astype(int)  # 1 = large, 0 = small
 
    gmap = dict(zip(snap["company_id"], snap["group"]))
    return np.array([gmap[c] for c in company_order], dtype=np.int64)
 
 
# ==============================================================================
# 8. STANDARDIZATION — fit on train, transform all (no leakage)
# ==============================================================================
 
def standardize_tensor(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    """
    X shape: (n, T, F). Flatten across time for fitting (year-agnostic scaling),
    per thesis Section 2.2.
    """
    n_tr, T, F = X_train.shape
    n_te = X_test.shape[0]
 
    scaler = StandardScaler()
    scaler.fit(X_train.reshape(-1, F))
 
    Xtr = scaler.transform(X_train.reshape(-1, F)).reshape(n_tr, T, F).astype(np.float32)
    Xte = scaler.transform(X_test.reshape(-1, F)).reshape(n_te, T, F).astype(np.float32)
    return Xtr, Xte, scaler
 
 
# ==============================================================================
# 9. SPLITS — k-Fold CV + Gap validation
# ==============================================================================
 
def make_kfold_splits(y: np.ndarray, k: int = K_FOLDS, seed: int = RANDOM_SEED
                      ) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Stratified k-fold indices list-of-(train_idx, test_idx)."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return [(tr, te) for tr, te in skf.split(np.zeros(len(y)), y)]
 
 
def apply_gap(train_idx: np.ndarray,
              gap_frac: float = GAP_FRACTION,
              seed: int = RANDOM_SEED) -> np.ndarray:
    """
    Thesis Section 7.2 — training set ke pehle 10% remove karo (temporal gap).
    Note: humare samples N-level hain (company-level), isliye index order
    ke hisaab se drop karte hain. Production me ye year-based hota hai.
    """
    rng = np.random.default_rng(seed)
    idx = np.array(train_idx).copy()
    rng.shuffle(idx)
    drop = int(np.floor(gap_frac * len(idx)))
    return idx[drop:]
 
 
# ==============================================================================
# 10. BAGGING BOOTSTRAP — 7 members
# ==============================================================================
 
def make_bagging_indices(
    n_train: int,
    n_members: int = ENSEMBLE_MEMBERS,
    seed: int = RANDOM_SEED,
) -> List[np.ndarray]:
    """Bootstrap indices (sampling with replacement) for each ensemble member."""
    rng = np.random.default_rng(seed)
    return [rng.integers(0, n_train, size=n_train) for _ in range(n_members)]
 
 
# ==============================================================================
# 11. FULL DATASET CONTAINER
# ==============================================================================
 
@dataclass
class AHNNDataset:
    """
    Poora data structure ek container me.
    Fields describe karte hain exactly kya-kya shape aur kya-kya content hona chahiye.
    """
    # Raw
    raw: pd.DataFrame                                # long format (N*T rows, RAW_COLUMNS)
    features: pd.DataFrame                           # long format (N*T rows, 3 + F cols)
    labels: pd.DataFrame                             # (N rows, [company_id, y, Z_label_year])
 
    # Tensors
    X: np.ndarray                                    # shape (N, T, F)
    y: np.ndarray                                    # shape (N,)
    company_order: List[int]                         # len N
 
    # Fairness
    group: np.ndarray                                # shape (N,), 0=small, 1=large
 
    # Metadata
    feature_names: List[str] = field(default_factory=lambda: list(FEATURE_NAMES))
    n_companies: int = N_COMPANIES
    n_years: int = T_YEARS
    n_features: int = F_FEATURES
 
    # Splits (populated later)
    kfold_splits: Optional[List[Tuple[np.ndarray, np.ndarray]]] = None
    bagging_indices: Optional[List[np.ndarray]] = None
 
    # ---------- Convenience methods ----------
    def summary(self) -> str:
        pos = int(self.y.sum())
        neg = len(self.y) - pos
        n_small = int((self.group == 0).sum())
        n_large = int((self.group == 1).sum())
        return (
            "============ AHNNDataset Summary ============\n"
            f"Shape          : X = {self.X.shape}, y = {self.y.shape}\n"
            f"N companies    : {self.n_companies}\n"
            f"T years        : {self.n_years}\n"
            f"F features     : {self.n_features}\n"
            f"Label balance  : default(y=1) = {pos}  |  safe(y=0) = {neg}  "
            f"(default rate = {pos/len(self.y):.1%})\n"
            f"Fairness group : small = {n_small}  |  large = {n_large}\n"
            f"k-Fold splits  : {K_FOLDS}\n"
            f"Gap fraction   : {GAP_FRACTION}\n"
            f"Ensemble size  : {ENSEMBLE_MEMBERS}\n"
            "=============================================="
        )
 
    def prepare_splits(self) -> None:
        """k-fold indices + bagging indices ko populate karta hai."""
        self.kfold_splits = make_kfold_splits(self.y)
        # Note: bagging indices ek particular train-size ke liye hote hain;
        # inhein per-fold call karna zyada proper hai. Yahan ek global default
        # (full-N) demonstrative ke liye.
        self.bagging_indices = make_bagging_indices(n_train=self.n_companies)
 
 
# ==============================================================================
# 12. END-TO-END PIPELINE FUNCTION
# ==============================================================================
 
def build_dataset_from_raw(raw: pd.DataFrame) -> AHNNDataset:
    """
    Raw balance-sheet DataFrame -> full AHNNDataset.
 
    Expected `raw` schema: columns = RAW_COLUMNS, rows = N*T = 62*8 = 496.
    """
    # Schema check
    missing = set(RAW_COLUMNS) - set(raw.columns)
    assert not missing, f"Missing raw columns: {missing}"
    assert len(raw) == N_COMPANIES * T_YEARS, \
        f"Expected {N_COMPANIES*T_YEARS} rows, got {len(raw)}"
 
    # Step 1 — compute ratios
    features = compute_features(raw)
 
    # Step 2 — build labels (company-level, based on last year's Z)
    labels = construct_labels(features, label_year=YEAR_END)
 
    # Step 3 — assemble tensors
    X, y, order = build_tensors(features, labels)
 
    # Step 4 — fairness groups
    group = assign_fairness_groups(raw, order, split_year=YEAR_END)
 
    ds = AHNNDataset(
        raw=raw,
        features=features,
        labels=labels,
        X=X,
        y=y,
        company_order=order,
        group=group,
    )
    ds.prepare_splits()
    return ds
 
 
# ==============================================================================
# 13. SYNTHETIC DATA GENERATOR — demonstration ke liye
# ==============================================================================
 
def generate_synthetic_raw(seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    N=62 companies x T=8 years ka realistic synthetic raw balance-sheet banata hai.
    Real data nahi hai, sirf structure validate karne ke liye.
    """
    rng = np.random.default_rng(seed)
    rows = []
    for cid in range(1, N_COMPANIES + 1):
        # company-level scale (some big, some small)
        scale = rng.lognormal(mean=10.0, sigma=1.2)          # ~22k median TA
        # company-level health (affects all years)
        health = rng.normal(0.0, 1.0)
        for y_idx, yr in enumerate(range(YEAR_START, YEAR_END + 1)):
            TA = scale * rng.uniform(0.9, 1.1)
            CA = TA * rng.uniform(0.3, 0.6)
            CL = CA * rng.uniform(0.4, 0.9)
            TL = TA * rng.uniform(0.3, 0.8 - 0.05 * health)  # healthy -> lower leverage
            Equity = TA - TL
            Inv = CA * rng.uniform(0.1, 0.5)
            Revenue = TA * rng.uniform(0.6, 1.4)
            GrossProfit = Revenue * rng.uniform(0.15, 0.45)
            EBIT = GrossProfit * rng.uniform(0.2, 0.7) + health * 0.05 * Revenue
            Interest = TL * rng.uniform(0.03, 0.09)
            NetIncome = EBIT - Interest - max(0, 0.25 * (EBIT - Interest))
            RE = NetIncome * rng.uniform(0.3, 1.0) * (y_idx + 1)
 
            rows.append({
                "company_id": cid,
                "company_name": f"Company_{cid:02d}",
                "year": yr,
                "CA": CA, "CL": CL, "TA": TA, "TL": TL,
                "Inventory": Inv, "Equity": Equity,
                "Revenue": Revenue, "GrossProfit": GrossProfit,
                "EBIT": EBIT, "NetIncome": NetIncome,
                "InterestExpense": Interest, "RetainedEarnings": RE,
            })
    return pd.DataFrame(rows, columns=RAW_COLUMNS)
 
 
# ==============================================================================
# 14. MAIN — demo run
# ==============================================================================
 
if __name__ == "__main__":
    print(">> Generating synthetic raw data...")
    raw = generate_synthetic_raw()
    print(f"   raw shape: {raw.shape}  (expected: {N_COMPANIES*T_YEARS} x {len(RAW_COLUMNS)})")
 
    print("\n>> Building AHNN dataset (features + labels + tensors + groups + splits)...")
    ds = build_dataset_from_raw(raw)
    print(ds.summary())
 
    print("\n>> Example — first fold shapes after standardization:")
    tr_idx, te_idx = ds.kfold_splits[0]
    X_tr, X_te = ds.X[tr_idx], ds.X[te_idx]
    y_tr, y_te = ds.y[tr_idx], ds.y[te_idx]
 
    X_tr_s, X_te_s, scaler = standardize_tensor(X_tr, X_te)
    print(f"   X_train: {X_tr_s.shape}   X_test: {X_te_s.shape}")
    print(f"   y_train: {y_tr.shape}     y_test: {y_te.shape}")
 
    gap_train_idx = apply_gap(tr_idx)
    print(f"\n>> Gap-validated training indices: {len(tr_idx)} -> {len(gap_train_idx)}")
 
    bags = make_bagging_indices(n_train=len(gap_train_idx))
    print(f">> Bagging bootstrap sets: {len(bags)} members, each with "
          f"{bags[0].shape[0]} indices (with replacement)")
 
    print("\n>> First 5 rows of engineered features:")
    print(ds.features.head().to_string(index=False))
 
    print("\n>> Label distribution:")
    print(ds.labels["y"].value_counts().to_string())
 
    print("\n>> Done. Data structure ready for AHNN training.")

>> Generating synthetic raw data...
   raw shape: (496, 15)  (expected: 496 x 15)

>> Building AHNN dataset (features + labels + tensors + groups + splits)...
============ AHNNDataset Summary ============
Shape          : X = (62, 8, 11), y = (62,)
N companies    : 62
T years        : 8
F features     : 11
Label balance  : default(y=1) = 13  |  safe(y=0) = 49  (default rate = 21.0%)
Fairness group : small = 31  |  large = 31
k-Fold splits  : 5
Gap fraction   : 0.1
Ensemble size  : 7

>> Example — first fold shapes after standardization:
   X_train: (49, 8, 11)   X_test: (13, 8, 11)
   y_train: (49,)     y_test: (13,)

>> Gap-validated training indices: 49 -> 45
>> Bagging bootstrap sets: 7 members, each with 45 indices (with replacement)

>> First 5 rows of engineered features:
 company_id company_name  year    WC_TA     RE_TA  EBIT_TA    EQ_TL     S_TA       CR       QR       NPM      GPM      Lev      ICR
          1   Company_01  2016 0.281548 -0.008780 0.034557 0.192545 1.228851 2.